In [ ]:
from collections.abc import Callable, Iterable
from functools import partial
from typing import Literal, NamedTuple

import numpy as np
import scipy.sparse
import scipy.sparse.linalg
import tqdm.notebook

from gacha_model import COND_PROB_5_STAR, COND_PROB_6_STAR
from plot_tools import draw_multi_pmf_cdf_fig, draw_pmf_cdf_fig
from random_variable import FiniteDist

In [ ]:
class 状态类(NamedTuple):
    六星水位: int
    五星水位: int
    已获取六星干员数量: int
    已获取五星干员数量: int


def 获取状态(*, 六星水位: int, 五星水位: int, 已获取六星干员数量: int, 已获取五星干员数量: int) -> 状态类:
    return 状态类(
        六星水位=六星水位,
        五星水位=五星水位,
        已获取六星干员数量=min(已获取六星干员数量, 已获取六星干员数量上限),
        已获取五星干员数量=min(已获取五星干员数量, 已获取五星干员数量上限),
    )


def 状态转移(起始状态: 状态类, *, 是第10抽: bool) -> list[tuple[状态类, float]]:
    转移概率列表: list[tuple[状态类, float]] = []

    起始六星水位 = 起始状态.六星水位
    起始五星水位 = 起始状态.五星水位
    起始已获取六星干员数量 = 起始状态.已获取六星干员数量
    起始已获取五星干员数量 = 起始状态.已获取五星干员数量

    # 计算抽到不同星级干员的概率
    六星概率 = COND_PROB_6_STAR[起始六星水位]
    if 是第10抽 and 起始五星水位 == 9:
        五星概率 = 1 - 六星概率
    else:
        五星概率 = np.clip(COND_PROB_5_STAR[起始五星水位], 0, 1 - 六星概率)
    四星或三星概率 = 1 - 六星概率 - 五星概率

    # 抽到 6★ 干员
    转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 已获取六星干员数量=起始已获取六星干员数量 + 1, 已获取五星干员数量=起始已获取五星干员数量), 六星概率))

    # 抽到 5★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 已获取六星干员数量=起始已获取六星干员数量, 已获取五星干员数量=起始已获取五星干员数量 + 1), 五星概率))

    # 抽到 4★ 及以下干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=起始五星水位 + 1, 已获取六星干员数量=起始已获取六星干员数量, 已获取五星干员数量=起始已获取五星干员数量), 四星或三星概率))

    转移概率列表 = [(目标状态, 概率) for 目标状态, 概率 in 转移概率列表 if 概率 > 0]

    assert np.isclose(sum(x[1] for x in 转移概率列表), 1)
    return 转移概率列表


def 构造状态转移矩阵(状态转移: Callable[[状态类], Iterable[tuple[状态类, float]]]) -> scipy.sparse.csr_array:
    状态数量 = len(状态列表)
    row = []
    col = []
    data = []
    for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
        转移概率列表 = 状态转移(起始状态)
        for 目标状态, 概率 in 转移概率列表:
            目标状态序号 = 状态索引[目标状态]
            row.append(起始状态序号)
            col.append(目标状态序号)
            data.append(概率)
    return scipy.sparse.csr_array((data, (row, col)), shape=(状态数量, 状态数量))


已获取六星干员数量上限 = 8
已获取五星干员数量上限 = 8


状态列表: list[状态类] = []
状态列表.extend(
    状态类(六星水位=六星水位, 五星水位=五星水位, 已获取六星干员数量=已获取六星干员数量, 已获取五星干员数量=已获取五星干员数量)
    for 六星水位 in range(len(COND_PROB_6_STAR))
    for 五星水位 in range(len(COND_PROB_5_STAR))
    for 已获取六星干员数量 in range(已获取六星干员数量上限 + 1)
    for 已获取五星干员数量 in range(已获取五星干员数量上限 + 1)
)
状态数量: int = len(状态列表)
状态索引: dict[状态类, int] = {状态: i for i, 状态 in enumerate(状态列表)}

状态转移矩阵_非第10抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=False))
状态转移矩阵_第10抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=True))

迭代次数 = 4096

初始状态 = 状态类(六星水位=0, 五星水位=0, 已获取六星干员数量=0, 已获取五星干员数量=0)
当前状态分布 = np.zeros(状态数量)
当前状态分布[状态索引[初始状态]] = 1

历史获得六星和五星干员数量联合分布 = np.zeros((迭代次数 + 1, 已获取六星干员数量上限 + 1, 已获取五星干员数量上限 + 1))

状态已获取六星干员数量数组 = np.fromiter((s.已获取六星干员数量 for s in 状态列表), dtype=int)
状态已获取五星干员数量数组 = np.fromiter((s.已获取五星干员数量 for s in 状态列表), dtype=int)

for i in tqdm.notebook.tqdm(range(迭代次数 + 1)):
    if i > 0:
        if i == 10:
            状态转移矩阵 = 状态转移矩阵_第10抽
        else:
            状态转移矩阵 = 状态转移矩阵_非第10抽
        当前状态分布 = 当前状态分布 @ 状态转移矩阵

    当前获得六星和五星干员数量联合分布 = np.zeros((已获取六星干员数量上限 + 1, 已获取五星干员数量上限 + 1))
    np.add.at(当前获得六星和五星干员数量联合分布, (状态已获取六星干员数量数组, 状态已获取五星干员数量数组), 当前状态分布)
    历史获得六星和五星干员数量联合分布[i] = 当前获得六星和五星干员数量联合分布

In [ ]:
# 在任意寻访中获得若干名任意 6★ 干员所需寻访次数的分布

dists = [FiniteDist([1])]

for n in range(1, 已获取六星干员数量上限 + 1):
    已获取六星干员数量矩阵 = np.arange(已获取六星干员数量上限 + 1)[None, :, None]
    已获取五星干员数量矩阵 = np.arange(已获取五星干员数量上限 + 1)[None, None, :]
    mask = 已获取六星干员数量矩阵 >= n
    cdf = np.sum(历史获得六星和五星干员数量联合分布 * mask, axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

dists[1] = FiniteDist(dists[1].pk[:99 + 1])

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 已获取六星干员数量上限 + 1)],
                       title="在任意寻访中获得若干名任意 6★ 干员所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在任意寻访中获得 1 名任意 6★ 干员所需寻访次数的分布",
                 x_max=None,
                 quantile_poses=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99, 1.00],
                 save=True)

In [ ]:
# 在任意寻访中获得若干名任意 5★ 及以上干员所需寻访次数的分布

dists = [FiniteDist([1])]

数量上限 = min(已获取六星干员数量上限, 已获取五星干员数量上限)

for n in range(1, 数量上限 + 1):
    已获取六星干员数量矩阵 = np.arange(已获取六星干员数量上限 + 1)[None, :, None]
    已获取五星干员数量矩阵 = np.arange(已获取五星干员数量上限 + 1)[None, None, :]
    mask = 已获取六星干员数量矩阵 + 已获取五星干员数量矩阵 >= n
    cdf = np.sum(历史获得六星和五星干员数量联合分布 * mask, axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

dists[1] = FiniteDist(dists[1].pk[:10 + 1])

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 数量上限 + 1)],
                       title="在任意寻访中获得若干名任意 5★ 及以上干员所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在任意寻访中获得 1 名任意 5★ 及以上干员所需寻访次数的分布",
                 x_max=None,
                 save=True)

In [ ]:
# 在任意寻访中获得若干名任意 5★ 干员所需寻访次数的分布

dists = [FiniteDist([1])]

for n in range(1, 已获取五星干员数量上限 + 1):
    已获取六星干员数量矩阵 = np.arange(已获取六星干员数量上限 + 1)[None, :, None]
    已获取五星干员数量矩阵 = np.arange(已获取五星干员数量上限 + 1)[None, None, :]
    mask = 已获取五星干员数量矩阵 >= n
    cdf = np.sum(历史获得六星和五星干员数量联合分布 * mask, axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 已获取五星干员数量上限 + 1)],
                       title="在任意寻访中获得若干名任意 5★ 干员所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在任意寻访中获得 1 名任意 5★ 干员所需寻访次数的分布",
                 x_max="auto",
                 save=True)

In [ ]:
def 打印状态分布(状态分布向量):
    for 状态序号, (状态, 概率) in enumerate(zip(状态列表, 状态分布向量)):
        if 概率 != 0:
            print(f"状态 {状态序号}: {状态}, 概率: {概率:.4%}")

In [ ]:
class 状态类(NamedTuple):
    六星水位: int
    五星水位: int
    上一次寻访的结果: Literal["6", "5", "4", "3"]


def 获取状态(*, 六星水位: int, 五星水位: int, 上一次寻访的结果: Literal["6", "5", "4", "3"]) -> 状态类:
    return 状态类(
        六星水位=六星水位,
        五星水位=五星水位,
        上一次寻访的结果=上一次寻访的结果,
    )


def 状态转移(起始状态: 状态类, *, 是第10抽: bool) -> list[tuple[状态类, float]]:
    转移概率列表: list[tuple[状态类, float]] = []

    起始六星水位 = 起始状态.六星水位
    起始五星水位 = 起始状态.五星水位

    # 计算抽到不同星级干员的概率
    六星概率 = COND_PROB_6_STAR[起始六星水位]
    if 是第10抽 and 起始五星水位 == 9:
        五星概率 = 1 - 六星概率
    else:
        五星概率 = np.clip(COND_PROB_5_STAR[起始五星水位], 0, 1 - 六星概率)
    四星概率 = np.clip(0.5, 0, 1 - 六星概率 - 五星概率)
    三星概率 = 1 - 六星概率 - 五星概率 - 四星概率

    # 抽到 6★ 干员
    转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 上一次寻访的结果="6"), 六星概率))

    # 抽到 5★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 上一次寻访的结果="5"), 五星概率))

    # 抽到 4★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=起始五星水位 + 1, 上一次寻访的结果="4"), 四星概率))

    # 抽到 3★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=起始五星水位 + 1, 上一次寻访的结果="3"), 三星概率))

    转移概率列表 = [(目标状态, 概率) for 目标状态, 概率 in 转移概率列表 if 概率 > 0]

    assert np.isclose(sum(x[1] for x in 转移概率列表), 1)
    return 转移概率列表


def 构造状态转移矩阵(状态转移: Callable[[状态类], Iterable[tuple[状态类, float]]]) -> scipy.sparse.csr_array:
    状态数量 = len(状态列表)
    row = []
    col = []
    data = []
    for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
        转移概率列表 = 状态转移(起始状态)
        for 目标状态, 概率 in 转移概率列表:
            目标状态序号 = 状态索引[目标状态]
            row.append(起始状态序号)
            col.append(目标状态序号)
            data.append(概率)
    return scipy.sparse.csr_array((data, (row, col)), shape=(状态数量, 状态数量))


def 计算平稳分布(状态转移矩阵: scipy.sparse.sparray | scipy.sparse.spmatrix, *, method: Literal["eigs", "lsqr", "lsmr", "spsolve", "bicg", "bicgstab", "gmres", "lgmres"] = "eigs") -> np.ndarray:
    assert 状态转移矩阵.ndim == 2 and 状态转移矩阵.shape[0] == 状态转移矩阵.shape[1], "状态转移矩阵必须是二维方阵"  # type: ignore
    状态数量 = 状态转移矩阵.shape[0]  # type: ignore

    if method == "eigs":
        eigenvalues, eigenvectors = scipy.sparse.linalg.eigs(状态转移矩阵.T, k=1, which='LM')  # type: ignore
        assert np.isclose(eigenvalues[0], 1), f"最大特征值不为 1: {eigenvalues[0]}"
        eigenvector = eigenvectors[:, 0].real  # type: ignore
        eigenvector /= eigenvector.sum()
        return eigenvector

    if method in ["lsqr", "lsmr"]:
        function = {
            "lsqr": scipy.sparse.linalg.lsqr,
            "lsmr": scipy.sparse.linalg.lsmr,
        }[method]

        矩阵 = scipy.sparse.vstack([状态转移矩阵.T - scipy.sparse.eye_array(状态数量), np.ones((1, 状态数量))], format="csc")  # type: ignore
        向量 = np.zeros(状态数量 + 1)
        向量[-1] = 1

        result = function(矩阵, 向量, atol=0, btol=0, conlim=0)
        x = result[0]
        return x

    elif method in ["spsolve", "bicg", "bicgstab", "gmres", "lgmres"]:
        function = {
            "spsolve": partial(scipy.sparse.linalg.spsolve),
            "bicg": partial(scipy.sparse.linalg.bicg, rtol=0),
            "bicgstab": partial(scipy.sparse.linalg.bicgstab, rtol=0),
            "gmres": partial(scipy.sparse.linalg.gmres, rtol=0),
            "lgmres": partial(scipy.sparse.linalg.lgmres, rtol=0),
        }[method]

        系数矩阵 = 状态转移矩阵.T - scipy.sparse.eye_array(状态数量)  # type: ignore
        矩阵 = scipy.sparse.vstack([系数矩阵[:-1], np.ones((1, 状态数量))], format="csc")
        向量 = np.zeros(状态数量)
        向量[-1] = 1

        result = function(矩阵, 向量)
        if isinstance(result, tuple):
            result = result[0]
        return result

    else:
        raise ValueError(f"未知的求解方法: {method}")


状态列表: list[状态类] = []
状态列表.extend(
    状态类(六星水位=六星水位, 五星水位=五星水位, 上一次寻访的结果=上一次寻访的结果)  # type: ignore
    for 六星水位 in range(len(COND_PROB_6_STAR))
    for 五星水位 in range(len(COND_PROB_5_STAR))
    for 上一次寻访的结果 in ["6", "5", "4", "3"]
)
状态数量: int = len(状态列表)
状态索引: dict[状态类, int] = {状态: i for i, 状态 in enumerate(状态列表)}

状态转移矩阵_非第10抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=False))
状态转移矩阵_第10抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=True))

状态上一次寻访的结果数组 = np.fromiter((s.上一次寻访的结果 for s in 状态列表), dtype='<U1')

# 计算平稳分布
平稳分布向量 = 计算平稳分布(状态转移矩阵_非第10抽, method="eigs")
平稳分布 = {}
for 星级 in ["6", "5", "4", "3"]:
    平稳分布[星级] = np.sum(平稳分布向量[状态上一次寻访的结果数组 == 星级])
for 星级, 概率 in 平稳分布.items():
    print(f"{星级}★：{概率:.4%}，{1/概率:.4f}")

In [ ]:
class 状态类(NamedTuple):
    六星水位: int
    五星水位: int
    上一次寻访的结果: Literal["6", "5", "4", "3"]


def 获取状态(*, 六星水位: int, 五星水位: int, 上一次寻访的结果: Literal["6", "5", "4", "3"]) -> 状态类:
    return 状态类(
        六星水位=六星水位,
        五星水位=五星水位,
        上一次寻访的结果=上一次寻访的结果,
    )


def 状态转移(起始状态: 状态类, *, 是第10抽: bool) -> list[tuple[状态类, float]]:
    转移概率列表: list[tuple[状态类, float]] = []

    起始六星水位 = 起始状态.六星水位
    起始五星水位 = 起始状态.五星水位

    # 计算抽到不同星级干员的概率
    六星概率 = COND_PROB_6_STAR[起始六星水位]
    if 是第10抽 and 起始五星水位 == 9:
        五星概率 = 1 - 六星概率
    else:
        五星概率 = np.clip(COND_PROB_5_STAR[起始五星水位], 0, 1 - 六星概率)
    四星概率 = np.clip(0.5, 0, 1 - 六星概率 - 五星概率)
    三星概率 = 1 - 六星概率 - 五星概率 - 四星概率

    # 抽到 6★ 干员
    转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 上一次寻访的结果="6"), 六星概率))

    # 抽到 5★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 上一次寻访的结果="5"), 五星概率))

    # 抽到 4★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=起始五星水位 + 1, 上一次寻访的结果="4"), 四星概率))

    # 抽到 3★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=起始五星水位 + 1, 上一次寻访的结果="3"), 三星概率))

    转移概率列表 = [(目标状态, 概率) for 目标状态, 概率 in 转移概率列表 if 概率 > 0]

    assert np.isclose(sum(x[1] for x in 转移概率列表), 1)
    return 转移概率列表


def 构造状态转移矩阵(状态转移: Callable[[状态类], Iterable[tuple[状态类, float]]]) -> scipy.sparse.csr_array:
    状态数量 = len(状态列表)
    row = []
    col = []
    data = []
    for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
        转移概率列表 = 状态转移(起始状态)
        for 目标状态, 概率 in 转移概率列表:
            目标状态序号 = 状态索引[目标状态]
            row.append(起始状态序号)
            col.append(目标状态序号)
            data.append(概率)
    return scipy.sparse.csr_array((data, (row, col)), shape=(状态数量, 状态数量))


def 计算平稳分布(状态转移矩阵: scipy.sparse.sparray | scipy.sparse.spmatrix, *, method: Literal["eigs", "lsqr", "lsmr", "spsolve", "bicg", "bicgstab", "gmres", "lgmres"] = "eigs") -> np.ndarray:
    assert 状态转移矩阵.ndim == 2 and 状态转移矩阵.shape[0] == 状态转移矩阵.shape[1], "状态转移矩阵必须是二维方阵"  # type: ignore
    状态数量 = 状态转移矩阵.shape[0]  # type: ignore

    if method == "eigs":
        eigenvalues, eigenvectors = scipy.sparse.linalg.eigs(状态转移矩阵.T, k=1, which='LM')  # type: ignore
        assert np.isclose(eigenvalues[0], 1), f"最大特征值不为 1: {eigenvalues[0]}"
        eigenvector = eigenvectors[:, 0].real  # type: ignore
        eigenvector /= eigenvector.sum()
        return eigenvector

    if method in ["lsqr", "lsmr"]:
        function = {
            "lsqr": scipy.sparse.linalg.lsqr,
            "lsmr": scipy.sparse.linalg.lsmr,
        }[method]

        矩阵 = scipy.sparse.vstack([状态转移矩阵.T - scipy.sparse.eye_array(状态数量), np.ones((1, 状态数量))], format="csc")  # type: ignore
        向量 = np.zeros(状态数量 + 1)
        向量[-1] = 1

        result = function(矩阵, 向量, atol=0, btol=0, conlim=0)
        x = result[0]
        return x

    elif method in ["spsolve", "bicg", "bicgstab", "gmres", "lgmres"]:
        function = {
            "spsolve": partial(scipy.sparse.linalg.spsolve),
            "bicg": partial(scipy.sparse.linalg.bicg, rtol=0),
            "bicgstab": partial(scipy.sparse.linalg.bicgstab, rtol=0),
            "gmres": partial(scipy.sparse.linalg.gmres, rtol=0),
            "lgmres": partial(scipy.sparse.linalg.lgmres, rtol=0),
        }[method]

        系数矩阵 = 状态转移矩阵.T - scipy.sparse.eye_array(状态数量)  # type: ignore
        矩阵 = scipy.sparse.vstack([系数矩阵[:-1], np.ones((1, 状态数量))], format="csc")
        向量 = np.zeros(状态数量)
        向量[-1] = 1

        result = function(矩阵, 向量)
        if isinstance(result, tuple):
            result = result[0]
        return result

    else:
        raise ValueError(f"未知的求解方法: {method}")


状态列表: list[状态类] = []
状态列表.extend(
    状态类(六星水位=六星水位, 五星水位=五星水位, 上一次寻访的结果=上一次寻访的结果)  # type: ignore
    for 六星水位 in range(len(COND_PROB_6_STAR))
    for 五星水位 in range(len(COND_PROB_5_STAR))
    for 上一次寻访的结果 in ["6", "5", "4", "3"]
)
状态数量: int = len(状态列表)
状态索引: dict[状态类, int] = {状态: i for i, 状态 in enumerate(状态列表)}

状态转移矩阵_非第10抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=False))
状态转移矩阵_第10抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=True))

状态上一次寻访的结果数组 = np.fromiter((s.上一次寻访的结果 for s in 状态列表), dtype='<U1')

# 计算平稳分布
平稳分布向量 = 计算平稳分布(状态转移矩阵_非第10抽, method="eigs")
平稳分布 = {}
for 星级 in ["6", "5", "4", "3"]:
    平稳分布[星级] = np.sum(平稳分布向量[状态上一次寻访的结果数组 == 星级])
print("平稳分布:")
for 星级, 概率 in 平稳分布.items():
    print(f"{星级}★：{概率:.4%}，{1/概率:.4f}")